<a href="https://colab.research.google.com/github/mimansa05/AI-LAB/blob/main/WEEK2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class VacuumEnvironment:
    def __init__(self):
        # 0 represents 'Clean', 1 represents 'Dirty'
        # The environment initializes both Room A and Room B as Dirty
        self.location_status = {'A': 1, 'B': 1}
        self.agent_location = 'A'  # Agent starts in Room A

    def is_dirty(self, location):
        return self.location_status[location] == 1

    def clean_location(self, location):
        self.location_status[location] = 0

    def move_agent(self, destination):
        self.agent_location = destination

    def display_state(self):
        status_map = {0: "Clean", 1: "Dirty"}
        print(f"Vacuum Location: [{self.agent_location}] | Room A: {status_map[self.location_status['A']]} | Room B: {status_map[self.location_status['B']]}")


class SimpleReflexAgent:
    def __init__(self):
        self.performance_score = 0  # Tracking action cost

    def program(self, environment):
        current_loc = environment.agent_location
        environment.display_state()

        # Rule 1: If current location is dirty, then Suck
        if environment.is_dirty(current_loc):
            print(f"-> Action: SUCK dust in Room {current_loc}")
            environment.clean_location(current_loc)
            self.performance_score += 1

        # Rule 2: If clean and in Room A, move Right
        elif current_loc == 'A':
            print("-> Action: Move RIGHT to Room B")
            environment.move_agent('B')
            self.performance_score += 1

        # Rule 3: If clean and in Room B, move Left
        elif current_loc == 'B':
            print("-> Action: Move LEFT to Room A")
            environment.move_agent('A')
            self.performance_score += 1


# --- Simulation Execution ---
if __name__ == "__main__":
    env = VacuumEnvironment()
    agent = SimpleReflexAgent()

    print("Initial Environment State:")
    step = 1

    # Keep acting until all rooms are Clean (state values are 0)
    while 1 in env.location_status.values():
        print(f"\n[Step {step}]")
        agent.program(env)
        step += 1

    print("\nFinal Goal State Achieved:")
    env.display_state()
    print(f"Total actions performed (Cost): {agent.performance_score}")

Initial Environment State:

[Step 1]
Vacuum Location: [A] | Room A: Dirty | Room B: Dirty
-> Action: SUCK dust in Room A

[Step 2]
Vacuum Location: [A] | Room A: Clean | Room B: Dirty
-> Action: Move RIGHT to Room B

[Step 3]
Vacuum Location: [B] | Room A: Clean | Room B: Dirty
-> Action: SUCK dust in Room B

Final Goal State Achieved:
Vacuum Location: [B] | Room A: Clean | Room B: Clean
Total actions performed (Cost): 3


In [ ]:
class PuzzleState:
    def __init__(self, board, parent=None, action=None, depth=0):
        self.board = board  # Tuple of 9 integers, 0 represents the blank space
        self.parent = parent
        self.action = action
        self.depth = depth

    def find_blank(self):
        return self.board.index(0)

    def get_neighbors(self):
        neighbors = []
        blank = self.find_blank()
        row, col = blank // 3, blank % 3

        # Define moves as (row_change, col_change, action_name)
        # Order: Up, Down, Left, Right (determines DFS exploration order)
        moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

        for dr, dc, act in moves:
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_blank = new_row * 3 + new_col
                # Create a new board layout by swapping tiles
                new_board = list(self.board)
                new_board[blank], new_board[new_blank] = new_board[new_blank], new_board[blank]
                neighbors.append(PuzzleState(tuple(new_board), self, act, self.depth + 1))
        return neighbors

    def is_goal(self, goal_board):
        return self.board == goal_board

    def print_board(self):
        for i in range(0, 9, 3):
            print(f"[{self.board[i]} {self.board[i+1]} {self.board[i+2]}]")
        print()


def solve_dfs(start_board, goal_board, max_depth=20):
    start_state = PuzzleState(start_board)

    # Stack stores (current_state)
    stack = [start_state]
    # Visited tracking to avoid infinite loops/graphs cycles
    visited = set()
    visited.add(start_board)

    nodes_expanded = 0

    while stack:
        current = stack.pop()  # DFS Pops from the end (LIFO)
        nodes_expanded += 1

        if current.is_goal(goal_board):
            print(f"🎉 Goal reached! Total nodes expanded: {nodes_expanded}")
            return current

        # Enforce depth limit to prevent deep infinite paths
        if current.depth < max_depth:
            # Reverse neighbors so they are explored in Up->Down->Left->Right order when popped
            for neighbor in reversed(current.get_neighbors()):
                if neighbor.board not in visited:
                    visited.add(neighbor.board)
                    stack.append(neighbor)

    print(f"❌ No solution found within depth limit of {max_depth}.")
    return None


def print_solution_path(solution_state):
    if not solution_state:
        return
    path = []
    current = solution_state
    while current:
        path.append(current)
        current = current.parent
    path.reverse()

    print(f"🏁 Solution Path ({len(path) - 1} moves):")
    for step, state in enumerate(path):
        if state.action:
            print(f"Step {step}: Move Blank '{state.action}'")
        else:
            print("Initial State:")
        state.print_board()


# --- Execution Configuration ---
# 0 represents the blank space
# A simple start state 2 moves away from the goal
START = (2, 8, 3,
        1, 6, 4,
        7, 0, 5)

GOAL = (1, 2, 3,
         8, 0, 4,
         7, 6, 5)

solution = solve_dfs(START, GOAL, max_depth=10)
print_solution_path(solution)


🎉 Goal reached! Total nodes expanded: 32
🏁 Solution Path (5 moves):
Initial State:
[2 8 3]
[1 6 4]
[7 0 5]

Step 1: Move Blank 'Up'
[2 8 3]
[1 0 4]
[7 6 5]

Step 2: Move Blank 'Up'
[2 0 3]
[1 8 4]
[7 6 5]

Step 3: Move Blank 'Left'
[0 2 3]
[1 8 4]
[7 6 5]

Step 4: Move Blank 'Down'
[1 2 3]
[0 8 4]
[7 6 5]

Step 5: Move Blank 'Right'
[1 2 3]
[8 0 4]
[7 6 5]



In [ ]:
class PuzzleState:
    def __init__(self, board, parent=None, action=None, depth=0):
        self.board = board  # Tuple of 9 integers, 0 represents the blank space
        self.parent = parent
        self.action = action
        self.depth = depth

    def find_blank(self):
        return self.board.index(0)

    def get_neighbors(self):
        neighbors = []
        blank = self.find_blank()
        row, col = blank // 3, blank % 3

        # Fixed priority order: Up, Down, Left, Right
        moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

        for dr, dc, act in moves:
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_blank = new_row * 3 + new_col
                new_board = list(self.board)
                new_board[blank], new_board[new_blank] = new_board[new_blank], new_board[blank]
                neighbors.append(PuzzleState(tuple(new_board), self, act, self.depth + 1))
        return neighbors

    def is_goal(self, goal_board):
        return self.board == goal_board

    def print_board(self):
        for i in range(0, 9, 3):
            print(f"  [{self.board[i]} {self.board[i+1]} {self.board[i+2]}]")


def solve_dfs_with_testing(start_board, goal_board, max_depth=10):
    start_state = PuzzleState(start_board)
    stack = [start_state]
    visited = set()
    visited.add(start_board)

    nodes_expanded = 0

    print("🚀 Starting Graph Evaluation Process...\n")

    while stack:
        current = stack.pop()  # DFS LIFO Pop
        nodes_expanded += 1

        print(f"📍 Node #{nodes_expanded} (Depth: {current.depth})")
        current.print_board()

        if current.is_goal(goal_board):
            print("\n🎉 GOAL STATE MATCH FOUND!")
            return current

        if current.depth < max_depth:
            # Re-verify boundaries dynamically to print status updates
            blank = current.find_blank()
            row, col = blank // 3, blank % 3
            moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

            test_results = []
            for dr, dc, act in moves:
                nr, nc = row + dr, col + dc
                if 0 <= nr < 3 and 0 <= nc < 3:
                    nb = nr * 3 + nc
                    test_board = list(current.board)
                    test_board[blank], test_board[nb] = test_board[nb], test_board[blank]

                    if tuple(test_board) in visited:
                        test_results.append(f"{act}: Blocked 🛑 (Already Visited)")
                    else:
                        test_results.append(f"{act}: Valid Path ✅ (Added to Stack)")
                else:
                    test_results.append(f"{act}: Out of Bounds ❌")

            print("🔍 Positional Tests performed:")
            for result in test_results:
                print(f"   ↳ {result}")

            # Reverse neighbors to maintain priority sequence inside stack
            for neighbor in reversed(current.get_neighbors()):
                if neighbor.board not in visited:
                    visited.add(neighbor.board)
                    stack.append(neighbor)
        else:
            print("⚠️ Maximum Depth Limit reached. Backtracking up the branch...")

        print("-" * 50)

    print(f"❌ Target goal could not be found within max depth of {max_depth}.")
    return None


# --- Target Execution Data Configuration ---
START = (2, 8, 3,
         1, 6, 4,
         7, 0, 5)

GOAL = (1, 2, 3,
        8, 0, 4,
        7, 6, 5)

solution = solve_dfs_with_testing(START, GOAL, max_depth=10)


🚀 Starting Graph Evaluation Process...

📍 Node #1 (Depth: 0)
  [2 8 3]
  [1 6 4]
  [7 0 5]
🔍 Positional Tests performed:
   ↳ Up: Valid Path ✅ (Added to Stack)
   ↳ Down: Out of Bounds ❌
   ↳ Left: Valid Path ✅ (Added to Stack)
   ↳ Right: Valid Path ✅ (Added to Stack)
--------------------------------------------------
📍 Node #2 (Depth: 1)
  [2 8 3]
  [1 0 4]
  [7 6 5]
🔍 Positional Tests performed:
   ↳ Up: Valid Path ✅ (Added to Stack)
   ↳ Down: Blocked 🛑 (Already Visited)
   ↳ Left: Valid Path ✅ (Added to Stack)
   ↳ Right: Valid Path ✅ (Added to Stack)
--------------------------------------------------
📍 Node #3 (Depth: 2)
  [2 0 3]
  [1 8 4]
  [7 6 5]
🔍 Positional Tests performed:
   ↳ Up: Out of Bounds ❌
   ↳ Down: Blocked 🛑 (Already Visited)
   ↳ Left: Valid Path ✅ (Added to Stack)
   ↳ Right: Valid Path ✅ (Added to Stack)
--------------------------------------------------
📍 Node #4 (Depth: 3)
  [0 2 3]
  [1 8 4]
  [7 6 5]
🔍 Positional Tests performed:
   ↳ Up: Out of Bounds ❌
 

In [ ]:
class PuzzleState:
    def __init__(self, board, parent=None, action=None, depth=0):
        self.board = board  # Tuple of 9 integers, 0 is the blank space
        self.parent = parent
        self.action = action
        self.depth = depth

    def find_blank(self):
        return self.board.index(0)

    def get_neighbors(self):
        neighbors = []
        blank = self.find_blank()
        row, col = blank // 3, blank % 3

        # Order: Up, Down, Left, Right
        moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

        for dr, dc, act in moves:
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_blank = new_row * 3 + new_col
                new_board = list(self.board)
                new_board[blank], new_board[new_blank] = new_board[new_blank], new_board[blank]
                neighbors.append(PuzzleState(tuple(new_board), self, act, self.depth + 1))
        return neighbors

    def print_board(self):
        for i in range(0, 9, 3):
            print(f"  [{self.board[i]} {self.board[i+1]} {self.board[i+2]}]")


def solve_dfs(start_board, goal_board, max_depth=10):
    start_state = PuzzleState(start_board)
    stack = [start_state]
    visited = set()
    visited.add(start_board)

    # Counter to track states explored
    states_explored = 0

    print("🏁 Starting DFS Solver Grid Exploration...\n")

    while stack:
        current = stack.pop()
        states_explored += 1

        # Track and print each single layout as it gets evaluated
        print(f"🔄 Explored State #{states_explored} (At Tree Depth: {current.depth})")
        if current.action:
            print(f"   Action Taken: Move blank '{current.action}'")
        else:
            print("   Action Taken: Root Initial State")
        current.print_board()
        print("-" * 40)

        # Check for goal match
        if current.board == goal_board:
            print(f"🎉 GOAL MATCH ENCOUNTERED!")
            print(f"📊 Total unique states explored during search: {states_explored}")
            return current

        # Push valid paths onto stack
        if current.depth < max_depth:
            for neighbor in reversed(current.get_neighbors()):
                if neighbor.board not in visited:
                    visited.add(neighbor.board)
                    stack.append(neighbor)

    print(f"❌ Target goal not found within depth limit.")
    return None


# --- Configuration Data ---
START = (2, 8, 3,
         1, 6, 4,
         7, 0, 5)

GOAL = (1, 2, 3,
        8, 0, 4,
        7, 6, 5)

# Run solver
solve_dfs(START, GOAL, max_depth=10)


In [ ]:
class PuzzleState:
    def __init__(self, board, parent=None, action=None, depth=0):
        self.board = board  # Tuple of 9 integers, 0 represents the blank space
        self.parent = parent
        self.action = action
        self.depth = depth

    def find_blank(self):
        return self.board.index(0)

    def is_goal(self, goal_board):
        return self.board == goal_board

    @staticmethod
    def format_board_row(board, row_idx):
        """Helper to format a single row of a board for side-by-side printing."""
        start = row_idx * 3
        return f"[{board[start]} {board[start+1]} {board[start+2]}]"


def solve_dfs_visual(start_board, goal_board, max_depth=10):
    start_state = PuzzleState(start_board)
    stack = [start_state]
    visited = set()
    visited.add(start_board)

    nodes_expanded = 0

    print("==========================================================================================")
    print("🎬 STARTING 8-PUZZLE EXPLORATION (DFS WITH ALL MOVEMENT POSSIBILITIES VISUALIZED)")
    print("==========================================================================================\n")

    while stack:
        current = stack.pop()
        nodes_expanded += 1

        print(f"📍 CURRENT NODE #{nodes_expanded} (Depth: {current.depth})")
        if current.action:
            print(f"👉 Generated by moving Blank '{current.action}'")
        else:
            print("👉 Starting Root Configuration")

        for i in range(3):
            print(f"   {PuzzleState.format_board_row(current.board, i)}")
        print()

        if current.is_goal(goal_board):
            print("🎯" * 40)
            print("🎉 SUCCESS! GOAL STATE MATCH FOUND!")
            print("🎯" * 40)
            return current

        if current.depth >= max_depth:
            print("⚠️ Depth limit reached. Backtracking up this branch...\n" + "-"*90 + "\n")
            continue

        # Evaluate all 4 possibilities side-by-side
        blank = current.find_blank()
        row, col = blank // 3, blank % 3

        # Priority Directions: Up, Down, Left, Right
        directions = [(-1, 0, 'UP ⬆️'), (1, 0, 'DOWN ⬇️'), (0, -1, 'LEFT ⬅️'), (0, 1, 'RIGHT ➡️')]

        # Store metadata for printing possibilities side-by-side
        headers = []
        statuses = []
        boards_to_print = []
        valid_neighbors_to_push = []

        for dr, dc, name in directions:
            nr, nc = row + dr, col + dc
            headers.append(f"{name:<13}")

            if 0 <= nr < 3 and 0 <= nc < 3:
                nb = nr * 3 + nc
                test_board = list(current.board)
                test_board[blank], test_board[nb] = test_board[nb], test_board[blank]
                test_board_tuple = tuple(test_board)

                boards_to_print.append(test_board_tuple)

                if test_board_tuple in visited:
                    statuses.append("❌ LOOP DETECTED ")
                else:
                    statuses.append("✅ VALID PATH    ")
                    # Create the state object for the stack
                    valid_neighbors_to_push.append(PuzzleState(test_board_tuple, current, name, current.depth + 1))
            else:
                boards_to_print.append(None)
                statuses.append("🚫 OUT OF BOUNDS ")

        # Print the testing matrix side-by-side
        print("🔍 EVALUATING ALL 4 POSSIBILITIES FOR THE NEXT MOVE:")
        print("   " + "       ".join(headers))
        print("   " + "       ".join(statuses))

        # Print the boards row by row
        for row_idx in range(3):
            row_str = "   "
            for b in boards_to_print:
                if b is not None:
                    row_str += PuzzleState.format_board_row(b, row_idx) + "       "
                else:
                    row_str += "[X X X]" + "       "
            print(row_str)
        print("\n" + "-"*90 + "\n")

        # Push valid paths onto stack.
        # Reversing them preserves the strict Up -> Down -> Left -> Right exploration sequence.
        for neighbor in reversed(valid_neighbors_to_push):
            visited.add(neighbor.board)
            stack.append(neighbor)

    print(f"❌ Execution finished. Goal not reachable within depth {max_depth}.")
    return None


# --- Target Data Configuration ---
START = (2, 8, 3,
         1, 6, 4,
         7, 0, 5)

GOAL = (1, 2, 3,
        8, 0, 4,
        7, 6, 5)

# Run the simulator
solve_dfs_visual(START, GOAL, max_depth=10)




🎬 STARTING 8-PUZZLE EXPLORATION (DFS WITH ALL MOVEMENT POSSIBILITIES VISUALIZED)

📍 CURRENT NODE #1 (Depth: 0)
👉 Starting Root Configuration
   [2 8 3]
   [1 6 4]
   [7 0 5]

🔍 EVALUATING ALL 4 POSSIBILITIES FOR THE NEXT MOVE:
   UP ⬆️               DOWN ⬇️             LEFT ⬅️             RIGHT ➡️     
   ✅ VALID PATH           🚫 OUT OF BOUNDS        ✅ VALID PATH           ✅ VALID PATH    
   [2 8 3]       [X X X]       [2 8 3]       [2 8 3]       
   [1 0 4]       [X X X]       [1 6 4]       [1 6 4]       
   [7 6 5]       [X X X]       [0 7 5]       [7 5 0]       

------------------------------------------------------------------------------------------

📍 CURRENT NODE #2 (Depth: 1)
👉 Generated by moving Blank 'UP ⬆️'
   [2 8 3]
   [1 0 4]
   [7 6 5]

🔍 EVALUATING ALL 4 POSSIBILITIES FOR THE NEXT MOVE:
   UP ⬆️               DOWN ⬇️             LEFT ⬅️             RIGHT ➡️     
   ✅ VALID PATH           ❌ LOOP DETECTED        ✅ VALID PATH           ✅ VALID PATH    
   [2 0 3]       

In [ ]:
class PuzzleState:
    def __init__(self, board, parent=None, action=None, depth=0):
        self.board = board  # Tuple of 9 integers, 0 is the blank space
        self.parent = parent
        self.action = action
        self.depth = depth

    def find_blank(self):
        return self.board.index(0)

    def get_neighbors(self):
        neighbors = []
        blank = self.find_blank()
        row, col = blank // 3, blank % 3

        # Order: Up, Down, Left, Right
        moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

        for dr, dc, act in moves:
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_blank = new_row * 3 + new_col
                new_board = list(self.board)
                new_board[blank], new_board[new_blank] = new_board[new_blank], new_board[blank]
                neighbors.append(PuzzleState(tuple(new_board), self, act, self.depth + 1))
        return neighbors

    def print_board(self):
        for i in range(0, 9, 3):
            print(f"  [{self.board[i]} {self.board[i+1]} {self.board[i+2]}]")


def solve_dfs(start_board, goal_board, max_depth=10):
    start_state = PuzzleState(start_board)
    stack = [start_state]
    visited = set()
    visited.add(start_board)

    # Counter to track states explored
    states_explored = 0

    print("🏁 Starting DFS Solver Grid Exploration...\n")

    while stack:
        current = stack.pop()
        states_explored += 1

        # Track and print each single layout as it gets evaluated
        print(f"🔄 Explored State #{states_explored} (At Tree Depth: {current.depth})")
        if current.action:
            print(f"   Action Taken: Move blank '{current.action}'")
        else:
            print("   Action Taken: Root Initial State")
        current.print_board()
        print("-" * 40)

        # Check for goal match
        if current.board == goal_board:
            print(f"🎉 GOAL MATCH ENCOUNTERED!")
            print(f"📊 Total unique states explored during search: {states_explored}")
            return current

        # Push valid paths onto stack
        if current.depth < max_depth:
            for neighbor in reversed(current.get_neighbors()):
                if neighbor.board not in visited:
                    visited.add(neighbor.board)
                    stack.append(neighbor)

    print(f"❌ Target goal not found within depth limit.")
    return None


# --- Configuration Data ---
START = (2, 8, 3,
         1, 6, 4,
         7, 0, 5)

GOAL = (1, 2, 3,
        8, 0, 4,
        7, 6, 5)

# Run solver
solve_dfs(START, GOAL, max_depth=10)


🏁 Starting DFS Solver Grid Exploration...

🔄 Explored State #1 (At Tree Depth: 0)
   Action Taken: Root Initial State
  [2 8 3]
  [1 6 4]
  [7 0 5]
----------------------------------------
🔄 Explored State #2 (At Tree Depth: 1)
   Action Taken: Move blank 'Up'
  [2 8 3]
  [1 0 4]
  [7 6 5]
----------------------------------------
🔄 Explored State #3 (At Tree Depth: 2)
   Action Taken: Move blank 'Up'
  [2 0 3]
  [1 8 4]
  [7 6 5]
----------------------------------------
🔄 Explored State #4 (At Tree Depth: 3)
   Action Taken: Move blank 'Left'
  [0 2 3]
  [1 8 4]
  [7 6 5]
----------------------------------------
🔄 Explored State #5 (At Tree Depth: 4)
   Action Taken: Move blank 'Down'
  [1 2 3]
  [0 8 4]
  [7 6 5]
----------------------------------------
🔄 Explored State #6 (At Tree Depth: 5)
   Action Taken: Move blank 'Down'
  [1 2 3]
  [7 8 4]
  [0 6 5]
----------------------------------------
🔄 Explored State #7 (At Tree Depth: 6)
   Action Taken: Move blank 'Right'
  [1 2 3]
  [7 

In [ ]:
class PuzzleState:
    def __init__(self, board, parent=None, action=None, depth=0):
        self.board = board  # Tuple of 9 integers, 0 is the blank space
        self.parent = parent
        self.action = action
        self.depth = depth

    def find_blank(self):
        return self.board.index(0)

    def get_neighbors(self):
        neighbors = []
        blank = self.find_blank()
        row, col = blank // 3, blank % 3

        # Execution priority sequence: Up, Down, Left, Right
        moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

        for dr, dc, act in moves:
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_blank = new_row * 3 + new_col
                new_board = list(self.board)
                new_board[blank], new_board[new_blank] = new_board[new_blank], new_board[blank]
                neighbors.append(PuzzleState(tuple(new_board), self, act, self.depth + 1))
        return neighbors


def solve_dfs_profiled(start_board, goal_board, max_depth=30):
    start_state = PuzzleState(start_board)
    stack = [start_state]
    visited = set()
    visited.add(start_board)

    # Complexity counters
    states_explored = 0
    peak_stack_size = 1

    print(f"🎬 Running Depth-Limited DFS (Max Depth Set to: {max_depth})...")
    print("Please wait, searching deep state branches...\n")

    while stack:
        current = stack.pop()
        states_explored += 1

        # Check for goal match
        if current.board == goal_board:
            print("==========================================================")
            print("🎉 GOAL STATE MATCH FOUND!")
            print("==========================================================")
            print(f"⏱️  TIME COMPLEXITY (Search Cost) : {states_explored} states expanded")
            print(f"💾 SPACE COMPLEXITY (Max Stack)   : {peak_stack_size} states at peak")
            print(f"💾 SPACE COMPLEXITY (Visited Set) : {len(visited)} states in memory")

            # Calculate final path cost
            path = []
            temp = current
            while temp.parent:
                path.append(temp.action)
                temp = temp.parent
            path.reverse()

            print(f"🏁 PATH COST (Solution Depth)     : {len(path)} moves")
            print(f"👉 Move Sequence                  : {path}")
            print("==========================================================")
            return current

        # Expand child nodes if within depth limit
        if current.depth < max_depth:
            for neighbor in reversed(current.get_neighbors()):
                if neighbor.board not in visited:
                    visited.add(neighbor.board)
                    stack.append(neighbor)

            # Dynamically check stack memory utilization peak
            if len(stack) > peak_stack_size:
                peak_stack_size = len(stack)

    print(f"❌ Puzzle unsolvable within a maximum limit of {max_depth} depth branches.")
    return None


# --- Target Data Configuration ---
START = (2, 8, 3,
         1, 6, 4,
         7, 0, 5)

GOAL = (1, 2, 3,
        8, 0, 4,
        7, 6, 5)

# Execute Profile Solver
solve_dfs_profiled(START, GOAL, max_depth=30)


🎬 Running Depth-Limited DFS (Max Depth Set to: 30)...
Please wait, searching deep state branches...

🎉 GOAL STATE MATCH FOUND!
⏱️  TIME COMPLEXITY (Search Cost) : 10354 states expanded
💾 SPACE COMPLEXITY (Max Stack)   : 28 states at peak
💾 SPACE COMPLEXITY (Visited Set) : 10359 states in memory
🏁 PATH COST (Solution Depth)     : 5 moves
👉 Move Sequence                  : ['Up', 'Up', 'Left', 'Down', 'Right']
